In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from typing import Any, List, Dict, Optional, Tuple
import time
import sys

sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score, KFold, learning_curve
from sklearn.metrics import r2_score, mean_squared_error
import xgboost as xgb

from src.data_loader import DataLoader
from src.preprocessing import DataPreprocessor
from src.feature_engineering import FeatureEngineer
from src.model import ModelTrainer

print("✅ All imports successful")

✅ All imports successful


In [2]:
# Load & Prepare Data 
loader = DataLoader("../config/config.yaml")
preprocessor = DataPreprocessor("../config/config.yaml")
engineer    = FeatureEngineer()

df = loader.load_data()
X, y = loader.split_features_target(df)
X = engineer.run_all(X, y)

df_full = X.copy()
df_full["SalePrice"] = y.values

X_train, X_test, y_train, y_test = preprocessor.run_pipeline(df_full)

print(f"✅ Data ready for training")
print(f"   X_train : {X_train.shape}")
print(f"   X_test  : {X_test.shape}")
print(f"   y_train : ${y_train.mean():,.0f} avg")

INFO:src.data_loader:✅ Config loaded from: ../config/config.yaml
INFO:src.data_loader:✅ DataLoader initialized successfully.
INFO:src.preprocessing:✅ DataPreprocessor initialized.
INFO:src.feature_engineering:✅ FeatureEngineer initialized.
INFO:src.data_loader:🔄 Loading data from: d:\STUDY\Programming\project\house-price-prediction\data\raw\housing_data.csv
INFO:src.data_loader:✅ Data loaded successfully. Shape: (90000, 28)
INFO:src.data_loader:✅ Split complete → X: (90000, 27), y: (90000,)
INFO:src.feature_engineering:==================================================
INFO:src.feature_engineering:🚀 RUNNING FEATURE ENGINEERING PIPELINE
INFO:src.feature_engineering:==================================================
INFO:src.feature_engineering:🔄 Adding age features...
INFO:src.feature_engineering:✅ Age features added.
INFO:src.feature_engineering:🔄 Adding area features...
INFO:src.feature_engineering:✅ Area features added.
INFO:src.feature_engineering:🔄 Adding quality features...
INFO:s

✅ Data ready for training
   X_train : (67327, 107)
   X_test  : (16832, 107)
   y_train : $299,153 avg


In [3]:
# Baseline — All Models 
trainer = ModelTrainer("../config/config.yaml")
results = trainer.train_baseline_models(X_train, y_train, X_test, y_test)

# Summary table
summary_df = pd.DataFrame({
    "Model"     : list(results.keys()),
    "Train R²"  : [results[m]["train_r2"]  for m in results],
    "Test R²"   : [results[m]["test_r2"]   for m in results],
    "RMSE ($)"  : [results[m]["test_rmse"] for m in results],
    "Time (s)"  : [results[m]["train_time"] for m in results],
}).sort_values("Test R²", ascending=False)

print("\n" + "=" * 65)
print("              MODEL COMPARISON RESULTS")
print("=" * 65)
print(summary_df.to_string(index=False))

INFO:src.model:✅ Initialized 7 models: ['linear_regression', 'ridge', 'lasso', 'decision_tree', 'random_forest', 'gradient_boosting', 'xgboost']
INFO:src.model:✅ ModelTrainer initialized.
INFO:src.model:=======================================================
INFO:src.model:🚀 BASELINE MODEL TRAINING
INFO:src.model:=======================================================



───────────────────────────────────────────────────────
Model                           R²         RMSE     Time
───────────────────────────────────────────────────────
linear_regression           0.8899 $     27,867    0.7s
ridge                       0.8899 $     27,864    0.1s
lasso                       0.8899 $     27,863   14.3s
decision_tree               0.7233 $     44,181    4.4s
random_forest               0.8684 $     30,466  123.4s
gradient_boosting           0.8839 $     28,622   64.1s


INFO:src.model:✅ Baseline training complete.


xgboost                     0.9000 $     26,555    4.3s
───────────────────────────────────────────────────────

              MODEL COMPARISON RESULTS
            Model  Train R²  Test R²  RMSE ($)  Time (s)
          xgboost    0.9330   0.9000  26554.97      4.33
linear_regression    0.8931   0.8899  27867.13      0.69
            ridge    0.8931   0.8899  27863.72      0.14
            lasso    0.8931   0.8899  27863.00     14.31
gradient_boosting    0.8907   0.8839  28622.20     64.09
    random_forest    0.9823   0.8684  30465.88    123.44
    decision_tree    1.0000   0.7233  44180.70      4.37


In [4]:
# Visual — Model Comparison 
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Baseline Model Comparison", fontsize=14, fontweight="bold")

colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(summary_df)))

# R² Score
axes[0].barh(summary_df["Model"], summary_df["Test R²"], color=colors)
axes[0].axvline(x=0.85, color="red", linestyle="--", label="Target (0.85)")
axes[0].set_xlabel("R² Score")
axes[0].set_title("R² Score (Higher = Better)")
axes[0].legend(fontsize=8)

# RMSE
axes[1].barh(summary_df["Model"], summary_df["RMSE ($)"], color=colors[::-1])
axes[1].set_xlabel("RMSE ($)")
axes[1].set_title("RMSE (Lower = Better)")

# Train vs Test R² (Overfitting Check)
x = np.arange(len(summary_df))
w = 0.35
axes[2].bar(x - w/2, summary_df["Train R²"], w,
            label="Train", color="#2563EB", alpha=0.8)
axes[2].bar(x + w/2, summary_df["Test R²"], w,
            label="Test", color="#10B981", alpha=0.8)
axes[2].set_xticks(x)
axes[2].set_xticklabels(summary_df["Model"], rotation=45, ha="right")
axes[2].set_title("Overfitting Check")
axes[2].legend()
axes[2].set_ylabel("R² Score")

plt.tight_layout()
plt.savefig("../reports/figures/model_comparison.png", dpi=150)
plt.show()

In [5]:
# Cross Validation
print("\n🔄 Running 5-Fold Cross Validation...")
cv_results = trainer.cross_validate_models(X_train, y_train)

# Visualize CV results
models_cv = list(cv_results.keys())
cv_means  = [cv_results[m]["cv_mean"] for m in models_cv]
cv_stds   = [cv_results[m]["cv_std"]  for m in models_cv]

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(models_cv))
bars = ax.bar(x, cv_means, yerr=cv_stds, capsize=5,
              color=plt.cm.Blues(np.linspace(0.5, 0.9, len(models_cv))),
              ecolor="gray", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(models_cv, rotation=30, ha="right")
ax.set_ylabel("Cross-Validation R² Score")
ax.set_title("5-Fold Cross Validation Results (Mean ± Std)",
             fontsize=13, fontweight="bold")
ax.axhline(y=0.85, color="red", linestyle="--", alpha=0.7, label="Target")
ax.legend()

for bar, val in zip(bars, cv_means):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.005,
            f"{val:.3f}", ha="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig("../reports/figures/cross_validation.png", dpi=150)
plt.show()

INFO:src.model:🔄 Running cross-validation...



🔄 Running 5-Fold Cross Validation...

────────────────────────────────────────────────────────────
Model                       CV Mean R²     CV Std
────────────────────────────────────────────────────────────
linear_regression               0.8928 ±  0.0022
ridge                           0.8928 ±  0.0022
lasso                           0.8928 ±  0.0022
decision_tree                   0.7252 ±  0.0051
random_forest                   0.8702 ±  0.0021
gradient_boosting               0.8865 ±  0.0021


INFO:src.model:✅ Cross-validation complete.


xgboost                         0.9035 ±  0.0027
────────────────────────────────────────────────────────────


In [6]:

# Learning Curve — Best Model
from sklearn.model_selection import learning_curve

best_model_instance = xgb.XGBRegressor(
    n_estimators=100, random_state=42, verbosity=0
)

train_sizes, train_scores, val_scores = learning_curve(
    best_model_instance,
    X_train, y_train,
    cv=5,
    scoring="r2",
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1,
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(train_sizes, train_mean, "o-", color="#2563EB",
        label="Training Score")
ax.fill_between(train_sizes,
                train_mean - train_std,
                train_mean + train_std,
                alpha=0.15, color="#2563EB")
ax.plot(train_sizes, val_mean, "o-", color="#10B981",
        label="Validation Score")
ax.fill_between(train_sizes,
                val_mean - val_std,
                val_mean + val_std,
                alpha=0.15, color="#10B981")
ax.axhline(y=0.85, color="red", linestyle="--", alpha=0.7,
           label="Target (0.85)")
ax.set_xlabel("Training Set Size")
ax.set_ylabel("R² Score")
ax.set_title("Learning Curve — XGBoost", fontsize=13, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../reports/figures/learning_curve.png", dpi=150)
plt.show()

In [7]:
# Hyperparameter Tuning
print("\n🔧 Starting Hyperparameter Tuning for XGBoost...")
print("   This may take a few minutes...")

tuned_model = trainer.tune_best_model(X_train, y_train, "xgboost")

# Evaluate tuned model
y_pred_tuned = tuned_model.predict(X_test)
tuned_r2   = r2_score(y_test, y_pred_tuned)
tuned_rmse = np.sqrt(mean_squared_error(y_test, y_pred_tuned))

print(f"\n✅ Tuned Model Results:")
print(f"   R² Score : {tuned_r2:.4f}")
print(f"   RMSE     : ${tuned_rmse:,.2f}")

# Compare before vs after tuning
base_r2   = results["xgboost"]["test_r2"]
base_rmse = results["xgboost"]["test_rmse"]

print(f"\n📈 Improvement from Tuning:")
print(f"   R² : {base_r2:.4f} → {tuned_r2:.4f} "
      f"(+{tuned_r2 - base_r2:.4f})")
print(f"   RMSE: ${base_rmse:,.0f} → ${tuned_rmse:,.0f} "
      f"(-${base_rmse - tuned_rmse:,.0f})")

INFO:src.model:🔄 Tuning hyperparameters for: xgboost



🔧 Starting Hyperparameter Tuning for XGBoost...
   This may take a few minutes...
Fitting 3 folds for each of 864 candidates, totalling 2592 fits


INFO:src.model:✅ Best params : {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 5, 'min_child_weight': 3, 'n_estimators': 300, 'reg_alpha': 0, 'reg_lambda': 1, 'subsample': 1.0}
   Best CV R² : 0.9140



✅ Tuned Model Results:
   R² Score : 0.9102
   RMSE     : $25,174.48

📈 Improvement from Tuning:
   R² : 0.9000 → 0.9102 (+0.0102)
   RMSE: $26,555 → $25,174 (-$1,380)


In [8]:
# Feature Importance 
if hasattr(tuned_model, "feature_importances_"):
    feat_imp = pd.DataFrame({
        "Feature"   : X_train.columns,
        "Importance": tuned_model.feature_importances_,
    }).sort_values("Importance", ascending=False).head(20)

    fig, ax = plt.subplots(figsize=(10, 8))
    colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(feat_imp)))[::-1]
    bars = ax.barh(feat_imp["Feature"][::-1],
                   feat_imp["Importance"][::-1], color=colors)
    ax.set_xlabel("Feature Importance Score")
    ax.set_title("Top 20 Feature Importances (XGBoost)",
                 fontsize=13, fontweight="bold")

    for bar, val in zip(bars, feat_imp["Importance"][::-1]):
        ax.text(bar.get_width() + 0.001,
                bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va="center", fontsize=8)

    plt.tight_layout()
    plt.savefig("../reports/figures/feature_importance.png", dpi=150)
    plt.show()

In [9]:
import os
os.makedirs("../models", exist_ok=True)

trainer.save_model(
    tuned_model,
    filepath="../models/best_model.pkl",
    scaler=preprocessor.scaler,
    scaler_path="../models/scaler.pkl",
)

print("\n🎉 Training Complete!")
print(f"   Model saved : ../models/best_model.pkl")
print(f"   Scaler saved: ../models/scaler.pkl")
print(f"   Final R²    : {tuned_r2:.4f}")
print(f"   Final RMSE  : ${tuned_rmse:,.2f}")

INFO:src.model:✅ Model saved to: ../models/best_model.pkl
INFO:src.model:✅ Scaler saved to: ../models/scaler.pkl



🎉 Training Complete!
   Model saved : ../models/best_model.pkl
   Scaler saved: ../models/scaler.pkl
   Final R²    : 0.9102
   Final RMSE  : $25,174.48
